# grok-012 · MiniCPM-V 农作物诊断（v11）

**模型**：`openbmb/MiniCPM-V-2_6-int4`（官方 T4/Colab int4 路径）

| 组件 | 版本 / 策略 |
|---|---|
| transformers | 4.40.0 |
| bitsandbytes | ≥0.44（cu12） |
| protobuf | ≥5.28（兼容 Kaggle TF 2.20） |
| TensorFlow | **强制禁用**（纯 PyTorch 推理，避免 AutoProcessor→TF 链） |
| flash_attn | stub |

> v10 失败根因：钉死 `protobuf==4.25.3` → TF 2.20 缺 `runtime_version`，MiniCPM remote code import `AutoProcessor` 时炸。


In [ ]:
import os, re, json, time, random, subprocess, sys
from pathlib import Path
from collections import defaultdict

# ---- pure torch path: never let transformers pull TF ----
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# fake flash_attn so check_imports passes
fake = Path("/tmp/fake_pkgs/flash_attn")
fake.mkdir(parents=True, exist_ok=True)
(fake / "__init__.py").write_text(
    "def flash_attn_func(*a,**k): raise RuntimeError('stub')\n"
    "def flash_attn_varlen_func(*a,**k): raise RuntimeError('stub')\n"
)
(fake / "bert_padding.py").write_text(
    "def index_first_axis(*a,**k): raise RuntimeError('stub')\n"
    "def pad_input(*a,**k): raise RuntimeError('stub')\n"
    "def unpad_input(*a,**k): raise RuntimeError('stub')\n"
)
sys.path.insert(0, "/tmp/fake_pkgs")

import torch
from PIL import Image
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "need GPU"
print("gpu", torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working_012")
OUT.mkdir(parents=True, exist_ok=True)
random.seed(42)
print("OUT", OUT)


In [ ]:
def pip(*pkgs, upgrade=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        cmd.append("-U")
    cmd.extend(pkgs)
    print("+", " ".join(cmd[3:]))
    subprocess.check_call(cmd)

# 1) protobuf first: TF 2.20 needs >=5.28 (v10 pin 4.25.3 broke runtime_version)
pip("protobuf>=5.28.0,<6", upgrade=True)

# 2) bitsandbytes (GPU int4)
pip("bitsandbytes>=0.44.0")

# 3) MiniCPM-V-2_6-int4 official-ish stack
pip(
    "transformers==4.40.0",
    "tokenizers==0.19.1",
    "accelerate==0.30.1",
    "sentencepiece==0.1.99",
)

# purge stale modules so reimport picks new wheels
for mod in list(sys.modules):
    if (
        mod == "transformers" or mod.startswith("transformers.")
        or mod == "bitsandbytes" or mod.startswith("bitsandbytes.")
        or mod == "google.protobuf" or mod.startswith("google.protobuf")
        or mod == "tensorflow" or mod.startswith("tensorflow.")
        or mod == "keras" or mod.startswith("keras.")
    ):
        del sys.modules[mod]

# verify protobuf has runtime_version
from google.protobuf import runtime_version as _rv  # noqa: F401
import google.protobuf as _pb
print("protobuf", getattr(_pb, "__version__", "?"))

import transformers.utils.import_utils as iu
# belt: transformers must never try TF
iu._tf_available = False
iu.is_tf_available = lambda: False
iu.is_flash_attn_2_available = lambda: False

import transformers
from transformers import AutoModel, AutoTokenizer
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.40"), transformers.__version__

import bitsandbytes as bnb
print("bitsandbytes", getattr(bnb, "__version__", "?"))
print("deps_ok")


In [ ]:
MODEL_ID = "openbmb/MiniCPM-V-2_6-int4"
print("loading", MODEL_ID)
t0 = time.perf_counter()
torch.cuda.empty_cache()
try:
    torch.cuda.reset_peak_memory_stats()
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model = model.eval()
try:
    model = model.cuda()
except Exception as e:
    print("cuda move note:", e)

load_s = time.perf_counter() - t0
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"loaded {load_s:.1f}s peak={peak:.2f}GB")
print("device", next(model.parameters()).device)
assert str(next(model.parameters()).device).startswith("cuda"), "model not on GPU"


In [ ]:
def clean_answer(s) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        try:
            s = "".join(list(s)) if hasattr(s, "__iter__") else str(s)
        except Exception:
            s = str(s)
    # strip common special / image tokens MiniCPM may emit
    s = re.sub(
        r"(?:<img>|</img>|<image>|</image>|<CLS>|<unk>|"
        r"\( <image>\./</image>\)|<quad>.*?</quad>)+",
        " ",
        s,
        flags=re.I | re.S,
    )
    return re.sub(r"\s+", " ", s).strip()


def is_bad(s: str) -> bool:
    t = clean_answer(s)
    if len(t) < 2:
        return True
    # pure punctuation / token junk
    if re.fullmatch(r"[\W_]+", t):
        return True
    return False


@torch.inference_mode()
def minicpm_chat(image, question: str, max_new_tokens: int = 128, sampling: bool = True):
    """Official 2.6 chat: content = [image(s), text], image=None kwarg."""
    if isinstance(image, list):
        content = list(image) + [question]
    else:
        content = [image, question]
    msgs = [{"role": "user", "content": content}]

    def _call(samp: bool):
        kwargs = dict(
            image=None,
            msgs=msgs,
            tokenizer=tokenizer,
            max_new_tokens=max_new_tokens,
            sampling=samp,
        )
        if samp:
            kwargs["temperature"] = 0.7
        t0 = time.perf_counter()
        res = model.chat(**kwargs)
        dt = time.perf_counter() - t0
        if not isinstance(res, str):
            try:
                res = "".join(list(res)) if hasattr(res, "__iter__") else str(res)
            except Exception:
                res = str(res)
        return clean_answer(res), dt

    # try sampling first (2.6 default quality); fall back greedy if junk/empty/nan path
    try:
        ans, dt = _call(sampling)
        if is_bad(ans) and sampling:
            print("  sampling bad, retry greedy…")
            ans, dt = _call(False)
        return ans, dt
    except Exception as e:
        print("  chat error:", type(e).__name__, e)
        if sampling:
            ans, dt = _call(False)
            return ans, dt
        raise


# solid-color smoke (should still produce a color word)
smoke = Image.new("RGB", (448, 448), (40, 140, 60))
ans, dt = minicpm_chat(smoke, "What is the main color of this image? Answer with one English word.")
print(f"[smoke {dt:.2f}s] {ans!r}")
assert not is_bad(ans), f"smoke failed: {ans!r}"
print("SMOKE_OK")


In [ ]:
def find_train_dirs(root: Path):
    hits = []
    if not root.exists():
        return hits
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() in {"train", "training"}:
            if sum(1 for d in p.iterdir() if d.is_dir()) >= 5:
                hits.append(p)
    return hits

train_dir = None
for r in [Path("/kaggle/input"), Path("./data")]:
    f = find_train_dirs(r)
    if f:
        train_dir = f[0]
        break
assert train_dir is not None, "PlantVillage train dir not found"
print("train_dir", train_dir)

class_to_paths = {}
for d in sorted(train_dir.iterdir()):
    if d.is_dir():
        imgs = [p for p in d.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        if imgs:
            class_to_paths[d.name] = imgs
class_names = sorted(class_to_paths)
print("n_classes", len(class_names))
assert len(class_names) >= 10

# real leaf smoke
p0 = random.choice(class_to_paths[class_names[0]])
im0 = Image.open(p0).convert("RGB")
im0.thumbnail((448, 448))
a0, d0 = minicpm_chat(im0, "Describe this plant leaf in one short English sentence.")
print(f"[real smoke {d0:.1f}s] {a0!r}")
assert not is_bad(a0), a0
print("REAL_SMOKE_OK")


In [ ]:
def parse_class_name(name: str):
    if "___" in name:
        c, d = name.split("___", 1)
    else:
        parts = name.split("_", 1)
        c, d = parts[0], parts[-1]
    return c, d, "healthy" in d.lower()


demo_rows = []
for cname in random.sample(class_names, k=min(3, len(class_names))):
    path = random.choice(class_to_paths[cname])
    im = Image.open(path).convert("RGB")
    im.thumbnail((448, 448))
    desc, dt1 = minicpm_chat(im, "Describe this plant leaf in one short English sentence.")
    jraw, dt2 = minicpm_chat(
        im,
        'Reply JSON only, no markdown: {"crop":"...","condition":"...","advice":"..."}',
        max_new_tokens=96,
    )
    print("==", cname)
    print(" desc:", desc)
    print(" json:", jraw)
    assert not is_bad(desc), desc
    demo_rows.append({
        "gold": cname,
        "path": str(path),
        "desc": desc,
        "raw_json": jraw,
        "sec": [dt1, dt2],
    })

(OUT / "demo_vqa.json").write_text(json.dumps(demo_rows, ensure_ascii=False, indent=2))
print("demo_vqa saved", len(demo_rows))


In [ ]:
N_EVAL = 8
rng = random.Random(42)
items = [(c, rng.choice(class_to_paths[c])) for c in rng.sample(class_names, k=min(N_EVAL, len(class_names)))]
PROMPT = (
    "Look at this crop leaf. Reply EXACTLY two lines:\n"
    "CROP: <name>\n"
    "DISEASE: <disease or healthy>"
)


def norm(s: str) -> str:
    s = (s or "").lower().replace("___", " ").replace("_", " ")
    s = re.sub(r"[^a-z0-9\u4e00-\u9fff ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def parse_two(text: str):
    crop = dis = ""
    for line in (text or "").splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            k, v = k.strip().lower(), v.strip()
            if "crop" in k:
                crop = v
            if "disease" in k or "condition" in k:
                dis = v
    # fallback: whole text if model ignored format
    if not crop and not dis and text:
        crop, dis = text, text
    return crop, dis


def score(gold, pc, pd):
    # empty prediction must NOT count as hit (v9 scoring bug: "" in s == True)
    if not norm(pc) and not norm(pd):
        return False, False, False
    gc, gd, gh = parse_class_name(gold)
    nc_g, nd_g, nc_p, nd_p = norm(gc), norm(gd), norm(pc), norm(pd)
    ch = bool(nc_p) and (
        (nc_g in nc_p) or (nc_p in nc_g) or (nc_g.split()[0] in nc_p)
    )
    if gh:
        dh = bool(nd_p) and ("healthy" in nd_p)
    else:
        keys = [t for t in re.split(r"[ _]+", gd) if len(t) > 2 and t.lower() != "healthy"]
        dh = bool(nd_p) and (
            any(norm(t) in nd_p for t in keys) or nd_g in nd_p or nd_p in nd_g
        )
    return bool(ch), bool(dh), bool(ch and dh)


rows = []
t0 = time.perf_counter()
for i, (g, p) in enumerate(items, 1):
    im = Image.open(p).convert("RGB")
    im.thumbnail((448, 448))
    ans, dt = minicpm_chat(im, PROMPT, max_new_tokens=64)
    pc, pd = parse_two(ans)
    ch, dh, both = score(g, pc, pd)
    rows.append({
        "gold": g,
        "pred_crop": pc,
        "pred_disease": pd,
        "crop_hit": ch,
        "disease_hit": dh,
        "both_hit": both,
        "sec": dt,
        "raw": ans,
    })
    print(f"[{i}] both={both} crop={ch} dis={dh} | {ans!r}")

n = len(rows)
metrics = {
    "n": n,
    "crop_acc": sum(r["crop_hit"] for r in rows) / n,
    "disease_acc": sum(r["disease_hit"] for r in rows) / n,
    "both_acc": sum(r["both_hit"] for r in rows) / n,
    "avg_sec": sum(r["sec"] for r in rows) / n,
    "total_sec": time.perf_counter() - t0,
    "model_id": MODEL_ID,
    "transformers": transformers.__version__,
    "n_bad": sum(1 for r in rows if is_bad(r["raw"])),
}
print(json.dumps(metrics, indent=2))
assert metrics["n_bad"] == 0, metrics
(OUT / "zero_shot_eval.json").write_text(
    json.dumps({"metrics": metrics, "rows": rows}, ensure_ascii=False, indent=2)
)
print("zero_shot saved")


In [ ]:
# multi-turn + multi-image compare
cname = random.choice([c for c in class_names if "healthy" not in c.lower()] or class_names)
path = random.choice(class_to_paths[cname])
im = Image.open(path).convert("RGB")
im.thumbnail((448, 448))

a1, _ = minicpm_chat(im, "What disease or condition? One short sentence.", 64)
msgs = [
    {"role": "user", "content": [im, "What disease or condition? One short sentence."]},
    {"role": "assistant", "content": a1},
    {"role": "user", "content": "List 3 short farmer actions."},
]
t0 = time.perf_counter()
a2 = model.chat(
    image=None,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.7,
    max_new_tokens=96,
)
a2 = clean_answer(a2)
if is_bad(a2):
    a2 = clean_answer(
        model.chat(
            image=None, msgs=msgs, tokenizer=tokenizer,
            sampling=False, max_new_tokens=96,
        )
    )
print("a1", a1)
print("a2", a2)
assert not is_bad(a1)
assert not is_bad(a2)
(OUT / "multiturn.json").write_text(
    json.dumps({"gold": cname, "a1": a1, "a2": a2, "sec": time.perf_counter() - t0},
               ensure_ascii=False, indent=2)
)

# two-image compare (2.6 multi-image)
c_a, c_b = random.sample(class_names, 2)
im_a = Image.open(random.choice(class_to_paths[c_a])).convert("RGB")
im_b = Image.open(random.choice(class_to_paths[c_b])).convert("RGB")
im_a.thumbnail((448, 448)); im_b.thumbnail((448, 448))
cmp_ans, cmp_dt = minicpm_chat(
    [im_a, im_b],
    "These are two crop leaves. In 2 short sentences: how do they differ in health?",
    max_new_tokens=96,
)
print("compare", cmp_ans)
assert not is_bad(cmp_ans)
(OUT / "multi_image_compare.json").write_text(
    json.dumps({"a": c_a, "b": c_b, "ans": cmp_ans, "sec": cmp_dt}, ensure_ascii=False, indent=2)
)

report = {
    "notebook": "grok-012-minicpmv-crop-vqa",
    "version": "v11",
    "model_id": MODEL_ID,
    "transformers": transformers.__version__,
    "load_seconds": load_s,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
    "device": str(next(model.parameters()).device),
    "zero_shot_metrics": metrics,
    "fix": "v11: protobuf>=5.28 + TF disabled; MiniCPM-V-2_6-int4 + transformers 4.40",
}
(OUT / "grok012_results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))
print("OK")
